# Week 2: Feature Engineering & Data Preprocessing for Machine Learning

**AnalystLab Africa · Data Science Internship** · **Author:** Molemi Molokoane

**Dataset:** [Heart Failure Prediction](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction) (fedesoriano, Kaggle, ODbL licence). 918 patients pooled from five hospital datasets (Cleveland, Hungary, Switzerland, Long Beach VA, Stalog).

**Goal:** turn the raw patient records into a clean, encoded, scaled, machine-learning-ready dataset for predicting whether a patient has heart disease, and document every preprocessing decision.

### Workflow
1. Business understanding
2. Data inspection
3. Exploratory visualisation
4. Data cleaning
5. Feature engineering
6. Train/test split
7. Outlier detection and treatment
8. Feature encoding
9. Feature scaling
10. Feature selection
11. Final machine-learning dataset
12. Answers to the business questions

## 0. Setup

The notebook runs both in Google Colab and locally. If `Data/heart.csv` is not next to the notebook (for example in a fresh Colab session), the raw file is read from the project's GitHub repository.

In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing and feature-selection tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# Display and plotting defaults
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42  # fixed seed so every run gives the same split and importances

# Paths: outputs go in folders next to the notebook
DATA_DIR, VIZ_DIR = "Data", "Visualizations"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)

LOCAL_CSV = os.path.join(DATA_DIR, "heart.csv")
GITHUB_CSV = ("https://raw.githubusercontent.com/Mo-lemi/AnalystLab-Africa-Data-Science/main/"
              "Week2_DataScience_Project/Data/heart.csv")


def save_fig(name):
    """Save the current figure to the Visualizations folder and show it."""
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, f"{name}.png"), bbox_inches="tight")
    plt.show()

## 1. Business Understanding

Cardiovascular disease is the world's leading cause of death, responsible for roughly 17.9 million deaths a year (WHO). Many of these deaths are preventable if high-risk patients are identified early and referred for further tests or treatment.

- **Business problem:** clinics need a fast, low-cost way to flag patients who are likely to have heart disease, using measurements already taken at a routine visit.
- **Objective:** prepare a dataset from which a classification model can predict heart disease reliably.
- **Target variable:** `HeartDisease` (1 = heart disease, 0 = normal).
- **Why predictive modelling:** a model can screen every patient consistently and instantly, helping clinicians decide who to prioritise for further tests.
- **Expected impact:** earlier diagnosis, better use of scarce specialist time and equipment, and fewer emergency admissions.

The full discussion is in `Reports/Business_Understanding_Report`.

## 2. Data Inspection

### 2.1 Load the dataset and display the first 10 rows

In [ ]:
# Read the raw CSV, locally if available, otherwise from GitHub
source = LOCAL_CSV if os.path.exists(LOCAL_CSV) else GITHUB_CSV
df_raw = pd.read_csv(source)
print(f"Loaded from: {source}")
df_raw.head(10)

### 2.2 Structure: rows, columns and data types

In [ ]:
print(f"Rows: {df_raw.shape[0]}   Columns: {df_raw.shape[1]}\n")
df_raw.info()

In [ ]:
# Group the columns by kind to plan the preprocessing
numeric_cols = df_raw.select_dtypes(include="number").columns.drop("HeartDisease").tolist()
categorical_cols = df_raw.select_dtypes(include="object").columns.tolist()
print("Numeric columns    :", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Target             : HeartDisease")
print("\nUnique values per column:")
df_raw.nunique()

### 2.3 Missing values

`isnull()` finds no gaps, but a value of **0** is impossible for `Cholesterol` or `RestingBP` in a living patient. These zeros are missing values in disguise, so they are counted separately.

In [ ]:
# Standard NaN check
missing = pd.DataFrame({
    "NaN count": df_raw.isnull().sum(),
    "Zero count": (df_raw == 0).sum(),
})
missing["Zero %"] = (missing["Zero count"] / len(df_raw) * 100).round(1)
missing

In [ ]:
# Zeros only mean "missing" for physiological measurements that cannot be 0
hidden_missing = {col: int((df_raw[col] == 0).sum()) for col in ["RestingBP", "Cholesterol"]}
print("Hidden missing values (recorded as 0):", hidden_missing)

# Does missing cholesterol relate to the target?
chol_zero = df_raw["Cholesterol"] == 0
print(f"\nHeart-disease rate when Cholesterol = 0 : {df_raw.loc[chol_zero, 'HeartDisease'].mean():.1%}")
print(f"Heart-disease rate when Cholesterol > 0 : {df_raw.loc[~chol_zero, 'HeartDisease'].mean():.1%}")

**Observation:** 172 patients (18.7%) have `Cholesterol = 0` and one patient has `RestingBP = 0`. Patients with a missing cholesterol reading have a much higher heart-disease rate, so the *fact that it is missing* carries information. These rows come mainly from the Switzerland hospital source, which did not record cholesterol. This is why section 4 keeps a missing-value flag rather than only filling in the gaps.

The 0 values in `FastingBS` (binary flag) and `Oldpeak` (ST depression of 0 mm is normal) are valid.

### 2.4 Duplicate records

In [ ]:
print("Exact duplicate rows:", df_raw.duplicated().sum())
print("Duplicates ignoring the target:", df_raw.drop(columns="HeartDisease").duplicated().sum())

### 2.5 Summary statistics

In [ ]:
df_raw.describe().T.round(2)

In [ ]:
df_raw.describe(include="object").T

In [ ]:
# Class balance of the target
target_counts = df_raw["HeartDisease"].value_counts().rename({0: "Normal (0)", 1: "Heart disease (1)"})
print(target_counts)
print(f"\nPositive class share: {df_raw['HeartDisease'].mean():.1%}")

**Inspection summary**

| Check | Result |
|---|---|
| Shape | 918 rows × 12 columns |
| Data types | 5 numeric (int/float), 5 text categorical, 1 binary flag (`FastingBS`) stored as int, binary target |
| Missing values | No NaNs, but 172 hidden zeros in `Cholesterol` and 1 in `RestingBP` |
| Duplicates | None |
| Target balance | 55% heart disease / 45% normal: mildly imbalanced, so stratified splitting is enough |
| Value ranges | `Oldpeak` has 13 negative values (ST elevation), which are clinically valid; `Cholesterol` reaches 603 mg/dl |

## 3. Exploratory Visualisation

These charts show how each feature relates to the target before any preprocessing. Throughout the notebook, **blue = Normal (0)** and **orange = Heart disease (1)**.

In [ ]:
# Colours chosen to be distinguishable for colour-blind readers
TARGET_COLORS = {0: "#2a78d6", 1: "#eb6834"}
TARGET_LABELS = {0: "Normal", 1: "Heart disease"}

# Figure 1: target distribution
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df_raw, x="HeartDisease", hue="HeartDisease", palette=TARGET_COLORS, legend=False, ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)
ax.set_xticks([0, 1], [TARGET_LABELS[0], TARGET_LABELS[1]])
ax.set_title("Figure 1: Target distribution (918 patients)")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Number of patients")
save_fig("01_target_distribution")

In [ ]:
# Figure 2: distribution of each numeric feature, split by diagnosis
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(data=df_raw, x=col, hue="HeartDisease", palette=TARGET_COLORS,
                 bins=30, element="step", ax=ax)
    ax.set_title(f"{col} by diagnosis")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of patients")
    ax.get_legend().set_title("Diagnosis")
    for text, label in zip(ax.get_legend().texts, ["Normal", "Heart disease"]):
        text.set_text(label)
axes.flat[-1].axis("off")  # five numeric features, six panels
fig.suptitle("Figure 2: Numeric feature distributions by diagnosis", fontsize=14)
save_fig("02_numeric_histograms")

**Observations**
- `Cholesterol` has a tall spike at 0. These are the 172 hidden missing values, and almost all of them are heart-disease patients.
- Heart-disease patients are older on average, reach a lower `MaxHR`, and have higher `Oldpeak` (ST depression).
- `Oldpeak` and `Cholesterol` are right-skewed; `RestingBP` has a single 0 value far from the rest.

In [ ]:
# Figure 3: heart-disease counts within each categorical feature
cat_features = ["Sex", "ChestPainType", "FastingBS", "RestingECG", "ExerciseAngina", "ST_Slope"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, cat_features):
    sns.countplot(data=df_raw, x=col, hue="HeartDisease", palette=TARGET_COLORS, ax=ax)
    ax.set_title(f"{col} by diagnosis")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of patients")
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles, ["Normal", "Heart disease"], title="Diagnosis")
fig.suptitle("Figure 3: Categorical features by diagnosis", fontsize=14)
save_fig("03_categorical_countplots")

In [ ]:
# Heart-disease rate in each category, to quantify Figure 3
for col in cat_features:
    rates = df_raw.groupby(col)["HeartDisease"].agg(rate="mean", patients="size")
    rates["rate"] = (rates["rate"] * 100).round(1).astype(str) + "%"
    print(f"\n{col}\n{rates}")

**Observations**
- `ST_Slope`: a *Flat* slope means 83% heart disease vs 20% for *Up*, the strongest single separator.
- `ChestPainType`: *ASY* (asymptomatic) patients have the highest rate (79%). Silent chest pain is a known warning sign.
- `ExerciseAngina = Y` (85%), `FastingBS = 1` (79%) and male sex (63% vs 26%) all raise the risk.
- `RestingECG` separates the classes only weakly (52% to 66%).

In [ ]:
# Figure 4: age against maximum heart rate
fig, ax = plt.subplots(figsize=(8, 5.5))
for label in [0, 1]:
    subset = df_raw[df_raw["HeartDisease"] == label]
    ax.scatter(subset["Age"], subset["MaxHR"], s=28, alpha=0.65, color=TARGET_COLORS[label],
               edgecolor="white", linewidth=0.6, label=TARGET_LABELS[label])
ages = np.arange(df_raw["Age"].min(), df_raw["Age"].max() + 1)
ax.plot(ages, 220 - ages, color="#52514e", linestyle="--", linewidth=1.5, label="Predicted max (220 - age)")
ax.set_title("Figure 4: Age vs maximum heart rate achieved")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Maximum heart rate (bpm)")
ax.legend(title="Diagnosis")
save_fig("04_age_vs_maxhr_scatter")

**Observation:** maximum heart rate falls with age, and heart-disease patients sit lower at every age. Most patients fall well below the age-predicted maximum (220 − age). This motivates an engineered feature: *the percentage of the predicted maximum that the patient reached* (section 5).

## 4. Data Cleaning

### 4.1 Rename columns for clarity
The original names mix abbreviations and styles (`Oldpeak`, `FastingBS`, `ST_Slope`). Descriptive snake_case names make the code and the final dataset easier to read.

In [ ]:
df = df_raw.copy()  # keep the raw data untouched for comparison
column_names = {
    "Age": "age",
    "Sex": "sex",
    "ChestPainType": "chest_pain_type",
    "RestingBP": "resting_bp",
    "Cholesterol": "cholesterol",
    "FastingBS": "fasting_blood_sugar",
    "RestingECG": "resting_ecg",
    "MaxHR": "max_heart_rate",
    "ExerciseAngina": "exercise_angina",
    "Oldpeak": "st_depression",
    "ST_Slope": "st_slope",
    "HeartDisease": "heart_disease",
}
df = df.rename(columns=column_names)
pd.DataFrame({"original": column_names.keys(), "renamed": column_names.values()})

### 4.2 Convert hidden missing values to NaN
A resting blood pressure or cholesterol of 0 is physically impossible, so these zeros are replaced with `NaN`. They will be **imputed after the train/test split** (section 6.2), using training-set medians only, so that no information from the test set leaks into the training data.

Because missing cholesterol is strongly linked to heart disease (section 2.3), a flag column `cholesterol_missing` records which values were missing before the zeros are replaced.

In [ ]:
# Flag first, then replace, so the information is not lost
df["cholesterol_missing"] = (df["cholesterol"] == 0).astype(int)
df[["resting_bp", "cholesterol"]] = df[["resting_bp", "cholesterol"]].replace(0, np.nan)

print("Missing values after conversion:")
print(df.isnull().sum()[df.isnull().sum() > 0])

### 4.3 Remove duplicates

In [ ]:
rows_before = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {rows_before - len(df)}  (rows remaining: {len(df)})")

No duplicate patients were found, so no rows were removed. The check stays in the pipeline so the notebook is safe to rerun on updated data.

### 4.4 Validate and correct data types
- Text columns become `category`, which documents that they are labels and uses less memory.
- `fasting_blood_sugar`, `cholesterol_missing` and the target are binary 0/1 integers.
- `resting_bp` and `cholesterol` became floats because they now contain `NaN`; they return to whole numbers after imputation.
- Value ranges are checked against clinically plausible limits.

In [ ]:
categorical_features = ["sex", "chest_pain_type", "resting_ecg", "exercise_angina", "st_slope"]
df[categorical_features] = df[categorical_features].astype("category")
df[["fasting_blood_sugar", "cholesterol_missing", "heart_disease"]] = (
    df[["fasting_blood_sugar", "cholesterol_missing", "heart_disease"]].astype(int))
df.dtypes

In [ ]:
# Range checks: every value should be clinically plausible
checks = {
    "age between 18 and 100": df["age"].between(18, 100).all(),
    "resting_bp between 60 and 250 (ignoring NaN)": df["resting_bp"].dropna().between(60, 250).all(),
    "cholesterol between 80 and 700 (ignoring NaN)": df["cholesterol"].dropna().between(80, 700).all(),
    "max_heart_rate between 50 and 220": df["max_heart_rate"].between(50, 220).all(),
    "st_depression between -3 and 7": df["st_depression"].between(-3, 7).all(),
    "binary columns contain only 0/1": df[["fasting_blood_sugar", "heart_disease"]].isin([0, 1]).all().all(),
}
pd.Series(checks, name="passes")

All ranges pass. The 13 negative `st_depression` values are kept: a negative value means ST *elevation*, which is a real clinical finding, not an error.

## 5. Feature Engineering

New features are added where they carry clinical meaning that the raw columns express only indirectly:

| New feature | Definition | Why |
|---|---|---|
| `cholesterol_missing` | 1 if cholesterol was not recorded | Missingness is linked to heart disease (created in 4.2) |
| `hr_pct_of_predicted` | `max_heart_rate / (220 − age)` | Max heart rate depends on age; this compares each patient with what is expected for their age. Failing to reach it (chronotropic incompetence) is a known cardiac warning sign |
| `has_st_depression` | 1 if `st_depression > 0` | 40% of patients have exactly 0, so whether any depression is present is a meaningful yes/no signal on its own |
| `age_group` | `<45`, `45–54`, `55–64`, `65+` | Clinically used age bands; tested for redundancy with `age` in section 10 |

**Irrelevant features:** the dataset has no ID, name or date columns, so nothing is removed on the grounds of being an identifier. Whether any medical feature should be dropped is decided with evidence in section 10 (feature selection).

In [ ]:
df["hr_pct_of_predicted"] = (df["max_heart_rate"] / (220 - df["age"])).round(3)
df["has_st_depression"] = (df["st_depression"] > 0).astype(int)
df["age_group"] = pd.cut(df["age"], bins=[0, 44, 54, 64, 120],
                         labels=["<45", "45-54", "55-64", "65+"], ordered=True)

df[["age", "max_heart_rate", "hr_pct_of_predicted", "st_depression", "has_st_depression", "age_group"]].head()

In [ ]:
# How do the new features relate to the target?
print("Heart-disease rate by age group:")
print(df.groupby("age_group", observed=True)["heart_disease"].mean().round(3))
print("\nHeart-disease rate by has_st_depression:")
print(df.groupby("has_st_depression")["heart_disease"].mean().round(3))
print("\nMean hr_pct_of_predicted by diagnosis:")
print(df.groupby("heart_disease")["hr_pct_of_predicted"].mean().round(3))

In [ ]:
# Figure 5: the engineered features against the target
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.boxplot(data=df, x="heart_disease", y="hr_pct_of_predicted", hue="heart_disease",
            palette=TARGET_COLORS, legend=False, ax=axes[0])
axes[0].set_xticks([0, 1], ["Normal", "Heart disease"])
axes[0].set_title("% of age-predicted max heart rate")
axes[0].set_xlabel("Diagnosis")
axes[0].set_ylabel("Max HR / (220 − age)")

rate_by_age = df.groupby("age_group", observed=True)["heart_disease"].mean().mul(100)
axes[1].bar(rate_by_age.index.astype(str), rate_by_age.values, color=TARGET_COLORS[1], width=0.6)
axes[1].bar_label(axes[1].containers[0], fmt="%.0f%%", padding=3)
axes[1].set_title("Heart-disease rate by age group")
axes[1].set_xlabel("Age group (years)")
axes[1].set_ylabel("Patients with heart disease (%)")
axes[1].set_ylim(0, 100)

rate_by_missing = df.groupby("cholesterol_missing")["heart_disease"].mean().mul(100)
axes[2].bar(["Recorded", "Missing"], rate_by_missing.values, color=TARGET_COLORS[1], width=0.6)
axes[2].bar_label(axes[2].containers[0], fmt="%.0f%%", padding=3)
axes[2].set_title("Heart-disease rate by cholesterol status")
axes[2].set_xlabel("Cholesterol value")
axes[2].set_ylabel("Patients with heart disease (%)")
axes[2].set_ylim(0, 100)

fig.suptitle("Figure 5: Engineered features vs diagnosis", fontsize=14)
save_fig("05_engineered_features")

## 6. Train/Test Split and Missing-Value Imputation

### 6.1 Stratified train/test split
The data is split **before** any step that learns from the data (imputation medians, outlier fences, scaling statistics). Those values are then learned from the training set only and applied unchanged to the test set. If they were computed on all 918 rows, information about the test patients would leak into training and make model scores look better than they really are.

- 80% train / 20% test, a common choice for a dataset of this size.
- `stratify=y` keeps the 55/45 class balance identical in both parts.

In [ ]:
TARGET = "heart_disease"
X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

pd.DataFrame({
    "rows": [len(X_train), len(X_test)],
    "heart disease %": [round(y_train.mean() * 100, 1), round(y_test.mean() * 100, 1)],
    "missing cholesterol": [X_train["cholesterol"].isna().sum(), X_test["cholesterol"].isna().sum()],
    "missing resting_bp": [X_train["resting_bp"].isna().sum(), X_test["resting_bp"].isna().sum()],
}, index=["train", "test"])

### 6.2 Impute missing values with the training median for each sex

| Option | Decision |
|---|---|
| Drop the 172 rows | Rejected: loses 19% of the data, and these patients are mostly positive cases, so dropping them would bias the sample |
| Mean | Rejected: cholesterol is right-skewed, so the mean is pulled up by extreme values |
| **Median per sex** | **Chosen:** robust to skew, and cholesterol and blood pressure differ between men and women |

The `cholesterol_missing` flag from section 4.2 keeps the information that the value was originally missing, so a model can still use it.

In [ ]:
impute_cols = ["resting_bp", "cholesterol"]

# Learn the medians from the training set only
train_medians = X_train.groupby("sex", observed=True)[impute_cols].median()
print("Training-set medians used for imputation:")
print(train_medians, "\n")


def impute_by_sex(frame):
    """Fill missing resting_bp and cholesterol with the training median for the patient's sex."""
    frame = frame.copy()
    sex = frame["sex"].astype(str)
    for col in impute_cols:
        frame[col] = frame[col].fillna(sex.map(train_medians[col].to_dict()))
        frame[col] = frame[col].round().astype(int)  # whole numbers again, as in the raw data
    return frame


X_train_before_impute = X_train.copy()  # kept for the before/after chart
X_train = impute_by_sex(X_train)
X_test = impute_by_sex(X_test)
print("Missing values left in train:", X_train.isna().sum().sum(), "| in test:", X_test.isna().sum().sum())

In [ ]:
# Figure 6: cholesterol distribution before and after imputation (training set)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
before = X_train_before_impute["cholesterol"].fillna(0)  # show the original zeros
axes[0].hist(before, bins=40, color=TARGET_COLORS[0], edgecolor="white")
axes[0].set_title("Before: missing values recorded as 0")
axes[1].hist(X_train["cholesterol"], bins=40, color=TARGET_COLORS[0], edgecolor="white")
axes[1].set_title("After: imputed with median per sex")
for ax in axes:
    ax.set_xlabel("Cholesterol (mg/dl)")
axes[0].set_ylabel("Number of patients (training set)")
fig.suptitle("Figure 6: Cholesterol before and after imputation", fontsize=14)
save_fig("06_cholesterol_imputation")

**Observation:** the impossible spike at 0 is gone. Imputation creates a smaller spike at the median, which is the expected side effect of median imputation; the `cholesterol_missing` flag lets a model tell these imputed values apart from real ones.

## 7. Outlier Detection and Treatment

### 7.1 Detect outliers with boxplots, the IQR method and Z-scores
Three methods are compared on the training set:
- **Boxplots** to see the outliers.
- **IQR method:** a value is an outlier if it lies below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. This does not assume a normal distribution, which suits skewed features like `st_depression`.
- **Z-score:** a value is an outlier if it lies more than 3 standard deviations from the mean. This assumes a roughly normal distribution, so it is used as a cross-check.

In [ ]:
continuous_cols = ["age", "resting_bp", "cholesterol", "max_heart_rate", "st_depression", "hr_pct_of_predicted"]

# IQR fences learned from the training set
q1 = X_train[continuous_cols].quantile(0.25)
q3 = X_train[continuous_cols].quantile(0.75)
iqr = q3 - q1
lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr

# Z-scores on the training set
z_scores = (X_train[continuous_cols] - X_train[continuous_cols].mean()) / X_train[continuous_cols].std()

outlier_summary = pd.DataFrame({
    "lower fence": lower_fence.round(2),
    "upper fence": upper_fence.round(2),
    "IQR outliers": ((X_train[continuous_cols] < lower_fence) | (X_train[continuous_cols] > upper_fence)).sum(),
    "Z-score outliers (|z|>3)": (z_scores.abs() > 3).sum(),
})
outlier_summary["IQR outliers %"] = (outlier_summary["IQR outliers"] / len(X_train) * 100).round(1)
outlier_summary

### 7.2 Treatment: cap (winsorise) at the IQR fences

| Option | Decision |
|---|---|
| Remove outlier rows | Rejected: the values are clinically possible (e.g. a cholesterol of 500 or blood pressure of 190 happen in real patients), and removing them would throw away real, often high-risk, patients from a small dataset |
| Keep unchanged | Rejected: extreme values would dominate distance- and gradient-based models (KNN, SVM, logistic regression) |
| **Cap at the IQR fences** | **Chosen:** every patient is kept, the value still ranks as "very high" or "very low", but it can no longer distort the scale |

The fences are learned on the training set and applied to both sets. The IQR method is preferred over the Z-score for setting the limits because several features are skewed.

In [ ]:
X_train_before_cap = X_train.copy()


def cap_outliers(frame):
    """Clip each continuous feature to the training-set IQR fences."""
    frame = frame.copy()
    for col in continuous_cols:
        frame[col] = frame[col].clip(lower=lower_fence[col], upper=upper_fence[col])
    return frame


X_train = cap_outliers(X_train)
X_test = cap_outliers(X_test)

capped_counts = (X_train_before_cap[continuous_cols] != X_train[continuous_cols]).sum()
print("Values capped in the training set:")
print(capped_counts)

In [ ]:
# Figure 7: boxplots before and after capping (training set)
fig, axes = plt.subplots(2, len(continuous_cols), figsize=(18, 7))
for i, col in enumerate(continuous_cols):
    for row, (frame, label) in enumerate([(X_train_before_cap, "before"), (X_train, "after")]):
        ax = axes[row, i]
        sns.boxplot(y=frame[col], color=TARGET_COLORS[0], width=0.5, ax=ax,
                    flierprops={"markerfacecolor": TARGET_COLORS[1], "markeredgecolor": "white", "markersize": 6})
        ax.set_title(f"{col}\n({label} capping)", fontsize=10)
        ax.set_ylabel(col if i == 0 else "")
        ax.set_xlabel("")
fig.suptitle("Figure 7: Outlier detection (top) and treatment by IQR capping (bottom)", fontsize=14)
save_fig("07_outlier_boxplots")

**Observation:** after capping no points remain outside the whiskers, while every patient stays in the dataset. `age` had no outliers and is unchanged.

### 7.3 Save the cleaned dataset
At this point the data is clean but still human-readable: renamed columns, missing values imputed, outliers capped, engineered features added, categories still as text and values in their original units. A `split` column records which rows belong to the training and test sets.

In [ ]:
cleaned = pd.concat([
    X_train.assign(heart_disease=y_train, split="train"),
    X_test.assign(heart_disease=y_test, split="test"),
]).sort_index()

cleaned_path = os.path.join(DATA_DIR, "heart_cleaned.csv")
cleaned.to_csv(cleaned_path, index_label="patient_id")
print(f"Saved {cleaned_path}: {cleaned.shape[0]} rows × {cleaned.shape[1]} columns")
cleaned.head()

## 8. Feature Encoding

Machine-learning models need numbers, so every text category is converted. The technique depends on the kind of variable:

| Variable | Categories | Technique | Why |
|---|---|---|---|
| `sex` | F, M | **Label encoding** (F=0, M=1) | Only two values, so a single 0/1 column loses nothing and implies no false order |
| `exercise_angina` | N, Y | **Label encoding** (N=0, Y=1) | Binary yes/no |
| `age_group` | <45, 45-54, 55-64, 65+ | **Ordinal encoding** (0 to 3) | The groups have a real order, which the codes preserve |
| `chest_pain_type` | ASY, ATA, NAP, TA | **One-hot encoding** | Nominal: no category is "more" than another. Label encoding (e.g. ASY=0 … TA=3) would invent an order that does not exist |
| `resting_ecg` | LVH, Normal, ST | **One-hot encoding** | Nominal |
| `st_slope` | Down, Flat, Up | **One-hot encoding** | Ordered as a slope, but the risk is *not* monotonic (Flat 83%, Down 78%, Up 20%), so an ordinal code would mislead linear models |

For one-hot encoding, one category per feature is dropped as the **reference level**, to avoid the *dummy-variable trap* (the columns would otherwise always sum to 1, which is perfect multicollinearity for linear models). The clinically "normal" or lowest-risk category is chosen as the reference so the remaining columns read as "risk compared with normal": `ATA` for chest pain, `Normal` for ECG and `Up` for ST slope.

All encoders are fitted on the training set.

In [ ]:
# Label encoding for the two binary categorical features
label_encoders = {}
for col in ["sex", "exercise_angina"]:
    encoder = LabelEncoder().fit(X_train[col].astype(str))
    label_encoders[col] = encoder
    print(f"{col}: {dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))}")

# One-hot encoding for the nominal features, with a chosen reference level
onehot_cols = ["chest_pain_type", "resting_ecg", "st_slope"]
reference_levels = {"chest_pain_type": "ATA", "resting_ecg": "Normal", "st_slope": "Up"}
onehot = OneHotEncoder(drop=[reference_levels[c] for c in onehot_cols], sparse_output=False)
onehot.fit(X_train[onehot_cols].astype(str))
print("\nOne-hot columns created:", list(onehot.get_feature_names_out(onehot_cols)))


def encode(frame):
    """Apply the fitted encoders and return an all-numeric DataFrame."""
    out = frame.drop(columns=onehot_cols).copy()
    for col, encoder in label_encoders.items():
        out[col] = encoder.transform(frame[col].astype(str))
    out["age_group"] = frame["age_group"].cat.codes  # ordered categories -> 0..3
    dummies = pd.DataFrame(onehot.transform(frame[onehot_cols].astype(str)).astype(int),
                           columns=onehot.get_feature_names_out(onehot_cols), index=frame.index)
    return pd.concat([out, dummies], axis=1)


X_train_enc = encode(X_train)
X_test_enc = encode(X_test)
print(f"\nEncoded training set: {X_train_enc.shape[1]} columns, all numeric: "
      f"{X_train_enc.dtypes.apply(pd.api.types.is_numeric_dtype).all()}")
X_train_enc.head()

## 9. Feature Scaling

The continuous features use very different units and ranges (cholesterol in the hundreds, `st_depression` between −2 and 4, `hr_pct_of_predicted` around 0.8). Models that rely on distances or gradients (KNN, SVM, logistic regression, neural networks) would let the large-valued features dominate. Tree-based models do not need scaling, but scaling does them no harm, so one scaled dataset serves every model.

Three scalers are compared on the training set:

In [ ]:
scalers = {"StandardScaler": StandardScaler(), "MinMaxScaler": MinMaxScaler(), "RobustScaler": RobustScaler()}
compare_cols = ["cholesterol", "st_depression"]

# Figure 8: the same features under each scaler
fig, axes = plt.subplots(len(compare_cols), 4, figsize=(18, 7))
for row, col in enumerate(compare_cols):
    axes[row, 0].hist(X_train_enc[col], bins=30, color=TARGET_COLORS[0], edgecolor="white")
    axes[row, 0].set_title(f"{col}: original")
    for i, (name, scaler) in enumerate(scalers.items(), start=1):
        scaled = scaler.fit_transform(X_train_enc[[col]]).ravel()
        axes[row, i].hist(scaled, bins=30, color=TARGET_COLORS[0], edgecolor="white")
        axes[row, i].set_title(f"{col}: {name}")
    for ax in axes[row]:
        ax.set_xlabel("Value")
    axes[row, 0].set_ylabel("Number of patients")
fig.suptitle("Figure 8: Effect of each scaler (shape is unchanged; only the axis moves)", fontsize=14)
save_fig("08_scaler_comparison")

In [ ]:
# Summary statistics of cholesterol under each scaler
rows = {}
for name, scaler in scalers.items():
    s = pd.Series(scaler.fit_transform(X_train_enc[["cholesterol"]]).ravel())
    rows[name] = {"mean": s.mean(), "std": s.std(), "median": s.median(),
                  "IQR": s.quantile(0.75) - s.quantile(0.25), "min": s.min(), "max": s.max()}
pd.DataFrame(rows).T.round(3)

**Choice: RobustScaler for the continuous features.**
- It centres on the **median** and divides by the **IQR**, so it is not pulled around by the remaining skew (`st_depression`) or by the spike of imputed cholesterol values at the median, both of which shift the mean and standard deviation that StandardScaler relies on.
- MinMaxScaler is rejected because it depends on the single smallest and largest values, which makes it the most sensitive to extremes.
- Capping (section 7) and RobustScaler work together: capping limits extreme values, RobustScaler makes the scale itself outlier-resistant.

**Not scaled:** the binary and one-hot columns (already 0/1, and scaling would make them harder to interpret) and `age_group` (0 to 3, already on a small range).

The scaler is fitted on the training set only and then applied to the test set.

In [ ]:
scaler = RobustScaler().fit(X_train_enc[continuous_cols])

X_train_scaled = X_train_enc.copy()
X_test_scaled = X_test_enc.copy()
X_train_scaled[continuous_cols] = scaler.transform(X_train_enc[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test_enc[continuous_cols])

print("Training set after scaling (medians ≈ 0, IQR ≈ 1):")
X_train_scaled[continuous_cols].describe().T.round(2)

## 10. Feature Selection

Three pieces of evidence, all computed on the training set:
1. **Correlation heatmap:** finds redundant pairs of features (they carry the same information) and shows each feature's correlation with the target.
2. **Mutual information:** measures how much knowing a feature reduces uncertainty about the target. It also captures non-linear relationships.
3. **Random-forest feature importance:** how much each feature helps a tree-based model separate the classes.

### 10.1 Correlation heatmap

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

train_full = X_train_scaled.assign(heart_disease=y_train)
corr = train_full.corr()

# Diverging colour scale: blue (negative) -> grey (0) -> orange (positive)
diverging = LinearSegmentedColormap.from_list("blue_grey_orange", ["#2a78d6", "#f0efec", "#eb6834"])

# Figure 9: correlation heatmap (lower triangle only, since the matrix is symmetric)
fig, ax = plt.subplots(figsize=(15, 12))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap=diverging, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f",
            annot_kws={"size": 8}, linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Pearson correlation coefficient", "shrink": 0.7}, ax=ax)
ax.set_title("Figure 9: Correlation heatmap of the encoded features and target (training set)", fontsize=14)
ax.set_xlabel("Feature")
ax.set_ylabel("Feature")
save_fig("09_correlation_heatmap")

In [ ]:
# Correlation of every feature with the target, strongest first
target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
target_corr.round(3).to_frame("correlation with heart_disease")

In [ ]:
# Highly correlated feature pairs (|r| > 0.7)
features = X_train_scaled.columns.tolist()
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(features) for b in features[i + 1:]
         if abs(corr.loc[a, b]) > 0.7]
high_corr_pairs = pd.DataFrame(pairs, columns=["feature A", "feature B", "correlation"]).round(3)
high_corr_pairs.sort_values("correlation", key=abs, ascending=False)

### 10.2 Feature importance: mutual information and random forest

In [ ]:
# Binary/ordinal columns are discrete; continuous ones are not
discrete_mask = [col not in continuous_cols for col in features]
mi = pd.Series(mutual_info_classif(X_train_scaled, y_train, discrete_features=discrete_mask,
                                   random_state=RANDOM_STATE), index=features)

forest = RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)
forest.fit(X_train_scaled, y_train)
rf_importance = pd.Series(forest.feature_importances_, index=features)

importance = pd.DataFrame({
    "mutual information": mi,
    "random forest": rf_importance,
    "|corr with target|": target_corr.abs(),
}).sort_values("random forest", ascending=False)
importance["MI rank"] = importance["mutual information"].rank(ascending=False).astype(int)
importance["RF rank"] = importance["random forest"].rank(ascending=False).astype(int)
importance.round(4)

In [ ]:
# Figure 10: feature importance from both methods
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
order = importance.index[::-1]  # most important at the top
axes[0].barh(order, importance.loc[order, "mutual information"], color=TARGET_COLORS[0], height=0.65)
axes[0].set_title("Mutual information with the target")
axes[0].set_xlabel("Mutual information (nats)")
axes[0].set_ylabel("Feature")
axes[1].barh(order, importance.loc[order, "random forest"], color=TARGET_COLORS[1], height=0.65)
axes[1].set_title("Random-forest importance (mean decrease in impurity)")
axes[1].set_xlabel("Importance (sums to 1)")
fig.suptitle("Figure 10: Feature importance (training set), sorted by random-forest importance", fontsize=14)
save_fig("10_feature_importance")

### 10.3 Decide which features to remove
Two transparent rules:
1. **Redundancy:** if two features correlate above |0.8|, they carry nearly the same information, so the one with the **lower mutual information** is removed.
2. **Low relevance:** a feature is removed if **both** methods agree it is uninformative (mutual information below 0.01 *and* random-forest importance below 0.02). Requiring both protects features that one method underrates.

In [ ]:
REDUNDANCY_THRESHOLD = 0.8
MI_MIN, RF_MIN = 0.01, 0.02

to_drop = {}
# Rule 1: redundant pairs, strongest correlation first
for a, b, r in sorted(pairs, key=lambda p: -abs(p[2])):
    if abs(r) > REDUNDANCY_THRESHOLD and a not in to_drop and b not in to_drop:
        weaker, stronger = (a, b) if mi[a] < mi[b] else (b, a)
        to_drop[weaker] = f"redundant with {stronger} (r = {r:.2f}), lower mutual information"

# Rule 2: features both methods rate as uninformative
for col in features:
    if col not in to_drop and mi[col] < MI_MIN and rf_importance[col] < RF_MIN:
        to_drop[col] = f"low relevance (MI = {mi[col]:.4f}, RF = {rf_importance[col]:.4f})"

selected_features = [col for col in features if col not in to_drop]
print(f"Features removed ({len(to_drop)}):")
for col, reason in to_drop.items():
    print(f"  - {col}: {reason}")
print(f"\nFeatures kept ({len(selected_features)}): {selected_features}")

### 10.4 Check that the selection does not lose predictive power
A quick 5-fold cross-validated logistic regression on the training set compares all features with the selected ones. This is a sanity check of the dataset, not the final model (modelling comes in a later week).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
baseline = LogisticRegression(max_iter=1000)
for name, cols in [("all features", features), ("selected features", selected_features)]:
    scores = cross_val_score(baseline, X_train_scaled[cols], y_train, cv=cv, scoring="roc_auc")
    print(f"{name:18s} ({len(cols):2d} columns): ROC AUC = {scores.mean():.3f} ± {scores.std():.3f}")

## 11. Final Machine-Learning Dataset

The final dataset contains only the selected features, encoded and scaled, plus the target. Three files are saved:
- `heart_ml_ready_train.csv` and `heart_ml_ready_test.csv`: ready to train and evaluate a model with no further preprocessing.
- `heart_ml_ready.csv`: both parts together with a `split` column, for reference.

In [ ]:
train_final = X_train_scaled[selected_features].assign(heart_disease=y_train)
test_final = X_test_scaled[selected_features].assign(heart_disease=y_test)
ml_ready = pd.concat([train_final.assign(split="train"), test_final.assign(split="test")]).sort_index()

train_final.to_csv(os.path.join(DATA_DIR, "heart_ml_ready_train.csv"), index_label="patient_id")
test_final.to_csv(os.path.join(DATA_DIR, "heart_ml_ready_test.csv"), index_label="patient_id")
ml_ready.to_csv(os.path.join(DATA_DIR, "heart_ml_ready.csv"), index_label="patient_id")

# Final readiness checks
readiness = {
    "no missing values": not ml_ready.isna().any().any(),
    "all feature columns numeric": train_final.drop(columns=TARGET).dtypes.apply(pd.api.types.is_numeric_dtype).all(),
    "no duplicate patient IDs": ml_ready.index.is_unique,
    "train and test have identical columns": list(train_final.columns) == list(test_final.columns),
    "class balance preserved (train vs test, within 2 points)": abs(y_train.mean() - y_test.mean()) < 0.02,
}
print(f"Train: {train_final.shape}   Test: {test_final.shape}\n")
pd.Series(readiness, name="passes")

In [ ]:
train_final.head(10)

### Preprocessing pipeline summary

| # | Step | Fitted on | Result |
|---|---|---|---|
| 1 | Rename columns to descriptive snake_case | n/a | 12 clear column names |
| 2 | Flag missing cholesterol, convert impossible zeros to NaN | n/a | 172 cholesterol + 1 blood-pressure values marked missing |
| 3 | Remove duplicates | n/a | 0 found |
| 4 | Correct data types, check value ranges | n/a | Categories typed, all ranges plausible |
| 5 | Engineer features | n/a | `hr_pct_of_predicted`, `has_st_depression`, `age_group`, `cholesterol_missing` |
| 6 | Stratified 80/20 train/test split | n/a | 734 train / 184 test |
| 7 | Impute missing values with median per sex | Train | No missing values |
| 8 | Cap outliers at IQR fences | Train | Extreme values limited, all patients kept |
| 9 | Label, ordinal and one-hot encoding | Train | All features numeric |
| 10 | RobustScaler on continuous features | Train | Median 0, IQR 1 |
| 11 | Feature selection (correlation, MI, random forest) | Train | Redundant and uninformative features removed |

## 13. Download the outputs (Google Colab only)
When the notebook runs in Colab, this cell zips the generated CSVs and charts and downloads them, so they can be added to the GitHub repository. Locally it does nothing.

In [ ]:
try:
    from google.colab import files  # only available inside Colab
    import shutil
    os.makedirs("week2_outputs", exist_ok=True)
    for folder in [DATA_DIR, VIZ_DIR]:
        shutil.copytree(folder, os.path.join("week2_outputs", folder), dirs_exist_ok=True)
    shutil.make_archive("week2_outputs", "zip", "week2_outputs")
    files.download("week2_outputs.zip")
except ImportError:
    print("Not running in Colab: outputs are already saved in Data/ and Visualizations/.")